# 🔒 Google Colab OpenVPN 服务器部署

这个notebook将帮助您在Google Colab上快速部署OpenVPN服务器。

## ⚠️ 重要提醒
- Colab会话有时间限制（通常12小时）
- 每次重启会话需要重新部署
- 建议使用付费的Colab Pro以获得更稳定的连接
- 请遵守Google Colab的使用条款

## 🚀 快速开始
1. 运行第一个代码块安装依赖
2. 运行第二个代码块配置OpenVPN
3. 运行第三个代码块启动服务器
4. 下载客户端配置文件

---

## 步骤 1: 安装依赖和准备环境

In [ ]:
# 安装OpenVPN和相关依赖
!apt-get update
!apt-get install -y openvpn easy-rsa iptables net-tools iproute2 curl wget

# 创建必要的目录
!mkdir -p /etc/openvpn/server/{config,keys,logs}
!mkdir -p /tmp/openvpn-client

# 检查TUN设备支持
print("\n🔍 检查TUN设备支持...")
!ls -la /dev/net/tun 2>/dev/null || echo "TUN设备不存在，正在创建..."

# 创建TUN设备（如果不存在）
!mkdir -p /dev/net
!mknod /dev/net/tun c 10 200 2>/dev/null || true
!chmod 666 /dev/net/tun

# 加载TUN模块
!modprobe tun 2>/dev/null || echo "TUN模块加载失败（可能已经加载）"

print("✅ 环境准备完成！")

# 显示系统信息
print("\n📊 系统信息:")
!uname -a
print(f"\n🌐 公网IP: {!curl -s ifconfig.me}")
print(f"\n🏠 内网IP: {!hostname -I}")

## 步骤 2: 生成OpenVPN配置和证书

In [ ]:
# 获取公网IP
PUBLIC_IP = !curl -s ifconfig.me
PUBLIC_IP = PUBLIC_IP[0]
print(f"🌐 检测到公网IP: {PUBLIC_IP}")

# 生成服务器配置
server_config = f"""
port 1194
proto udp
dev tun
ca ca.crt
cert server.crt
key server.key
dh dh.pem
auth SHA256
cipher AES-256-CBC
server 10.8.0.0 255.255.255.0
ifconfig-pool-persist ipp.txt
push "redirect-gateway def1 bypass-dhcp"
push "dhcp-option DNS 8.8.8.8"
push "dhcp-option DNS 8.8.4.4"
keepalive 10 120
tls-auth ta.key 0
key-direction 0
cipher AES-256-GCM
auth SHA256
user nobody
group nogroup
persist-key
persist-tun
status openvpn-status.log
verb 3
explicit-exit-notify 1
"""

# 写入服务器配置
with open('/etc/openvpn/server/config/server.conf', 'w') as f:
    f.write(server_config.strip())

print("✅ 服务器配置文件已生成")

# 生成证书和密钥
print("\n🔐 生成证书和密钥...")

# 设置Easy-RSA环境
!cd /usr/share/easy-rsa && ./easyrsa init-pki
!cd /usr/share/easy-rsa && echo 'yes' | ./easyrsa build-ca nopass
!cd /usr/share/easy-rsa && ./easyrsa build-server-full server nopass
!cd /usr/share/easy-rsa && ./easyrsa gen-dh
!cd /usr/share/easy-rsa && openvpn --genkey --secret ta.key

# 复制证书到OpenVPN目录
!cp /usr/share/easy-rsa/pki/ca.crt /etc/openvpn/server/keys/
!cp /usr/share/easy-rsa/pki/issued/server.crt /etc/openvpn/server/keys/
!cp /usr/share/easy-rsa/pki/private/server.key /etc/openvpn/server/keys/
!cp /usr/share/easy-rsa/pki/dh.pem /etc/openvpn/server/keys/
!cp /usr/share/easy-rsa/ta.key /etc/openvpn/server/keys/

print("✅ 证书和密钥生成完成")

# 生成客户端证书
print("\n👤 生成客户端证书...")
!cd /usr/share/easy-rsa && ./easyrsa build-client-full client1 nopass

# 生成客户端配置
client_config = f"""
client
dev tun
proto udp
remote {PUBLIC_IP} 1194
resolv-retry infinite
nobind
persist-key
persist-tun
remote-cert-tls server
cipher AES-256-GCM
auth SHA256
key-direction 1
verb 3

<ca>
{open('/etc/openvpn/server/keys/ca.crt').read()}
</ca>

<cert>
{open('/usr/share/easy-rsa/pki/issued/client1.crt').read()}
</cert>

<key>
{open('/usr/share/easy-rsa/pki/private/client1.key').read()}
</key>

<tls-auth>
{open('/etc/openvpn/server/keys/ta.key').read()}
</tls-auth>
"""

# 写入客户端配置
with open('/tmp/openvpn-client/client1.ovpn', 'w') as f:
    f.write(client_config.strip())

print("✅ 客户端配置文件已生成")
print(f"📁 客户端配置文件位置: /tmp/openvpn-client/client1.ovpn")

## 步骤 3: 启动OpenVPN服务器

In [ ]:
# 配置iptables规则
print("🔧 配置防火墙规则...")
!iptables -t nat -A POSTROUTING -s 10.8.0.0/24 -o eth0 -j MASQUERADE
!iptables -A INPUT -p udp --dport 1194 -j ACCEPT
!iptables -A FORWARD -i tun0 -j ACCEPT
!iptables -A FORWARD -o tun0 -j ACCEPT

# 启用IP转发
!echo 1 > /proc/sys/net/ipv4/ip_forward

print("✅ 防火墙规则配置完成")

# 启动OpenVPN服务器
print("\n🚀 启动OpenVPN服务器...")
!cd /etc/openvpn/server && openvpn --config config/server.conf --daemon

# 等待服务器启动
import time
time.sleep(3)

# 检查服务器状态
print("\n📊 检查服务器状态...")
!ps aux | grep openvpn | grep -v grep

# 检查端口监听
print("\n🔍 检查端口监听状态...")
!netstat -tulnp | grep :1194

# 检查tun接口
print("\n🌐 检查TUN接口...")
!ip addr show tun0 2>/dev/null || echo "TUN接口尚未创建，请稍等..."

print("\n✅ OpenVPN服务器启动完成！")
print(f"🌐 服务器地址: {PUBLIC_IP}")
print("🔌 端口: 1194 (UDP)")
print("📱 请下载客户端配置文件进行连接")

## 步骤 4: 下载客户端配置文件

In [ ]:
# 显示客户端配置文件内容
print("📄 客户端配置文件内容:")
print("=" * 50)
with open('/tmp/openvpn-client/client1.ovpn', 'r') as f:
    print(f.read())
print("=" * 50)

# 创建下载链接
from google.colab import files
import os

print("\n📥 下载客户端配置文件:")
files.download('/tmp/openvpn-client/client1.ovpn')

print("\n📋 使用说明:")
print("1. 下载client1.ovpn文件到您的设备")
print("2. 安装OpenVPN客户端软件")
print("3. 导入client1.ovpn配置文件")
print("4. 连接到VPN服务器")
print(f"\n🌐 服务器信息:")
print(f"   地址: {PUBLIC_IP}")
print(f"   端口: 1194")
print(f"   协议: UDP")

## 📊 服务器监控

In [ ]:
# 实时监控服务器状态
import time
from IPython.display import clear_output

def monitor_server():
    while True:
        clear_output(wait=True)
        print("🔒 OpenVPN 服务器监控")
        print("=" * 40)
        
        # 检查进程
        print("\n📊 进程状态:")
        !ps aux | grep openvpn | grep -v grep
        
        # 检查端口
        print("\n🔌 端口状态:")
        !netstat -tulnp | grep :1194
        
        # 检查TUN接口
        print("\n🌐 TUN接口状态:")
        !ip addr show tun0 2>/dev/null || echo "TUN接口未创建"
        
        # 检查连接状态
        print("\n👥 连接状态:")
        !cat /etc/openvpn/server/logs/openvpn-status.log 2>/dev/null | tail -10 || echo "状态日志不存在"
        
        # 显示日志
        print("\n📝 最新日志:")
        !tail -5 /etc/openvpn/server/logs/openvpn.log 2>/dev/null || echo "日志文件不存在"
        
        print(f"\n⏰ 更新时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n按 Ctrl+C 停止监控")
        
        time.sleep(5)

# 启动监控（可选）
print("🔍 启动服务器监控...")
print("按 Ctrl+C 停止监控")
try:
    monitor_server()
except KeyboardInterrupt:
    print("\n✅ 监控已停止")

## 🛠️ 服务器管理

In [ ]:
# 服务器管理函数
def stop_server():
    """停止OpenVPN服务器"""
    print("🛑 停止OpenVPN服务器...")
    !pkill openvpn
    time.sleep(2)
    print("✅ 服务器已停止")

def restart_server():
    """重启OpenVPN服务器"""
    print("🔄 重启OpenVPN服务器...")
    stop_server()
    time.sleep(2)
    !cd /etc/openvpn/server && openvpn --config config/server.conf --daemon
    time.sleep(3)
    print("✅ 服务器已重启")

def show_status():
    """显示服务器状态"""
    print("📊 服务器状态:")
    !ps aux | grep openvpn | grep -v grep
    print("\n🔌 端口状态:")
    !netstat -tulnp | grep :1194

def show_logs():
    """显示服务器日志"""
    print("📝 服务器日志:")
    !tail -20 /etc/openvpn/server/logs/openvpn.log 2>/dev/null || echo "日志文件不存在"

# 管理菜单
print("🛠️ OpenVPN 服务器管理")
print("=" * 30)
print("1. 显示状态")
print("2. 显示日志")
print("3. 重启服务器")
print("4. 停止服务器")
print("5. 生成新客户端证书")
print("=" * 30)

# 示例：显示状态
print("\n📊 当前服务器状态:")
show_status()

## 🔧 故障排除

In [ ]:
# 故障排除工具
def check_tun_device():
    """检查TUN设备"""
    print("🔍 检查TUN设备:")
    !ls -la /dev/net/tun
    !lsmod | grep tun
    !ip tuntap show

def check_firewall():
    """检查防火墙规则"""
    print("🔧 检查防火墙规则:")
    !iptables -L -n | grep -E "(1194|tun0|10.8.0.0)"

def check_network():
    """检查网络配置"""
    print("🌐 检查网络配置:")
    !ip route show
    !cat /proc/sys/net/ipv4/ip_forward

def check_certificates():
    """检查证书文件"""
    print("🔐 检查证书文件:")
    !ls -la /etc/openvpn/server/keys/

# 运行诊断
print("🔧 运行系统诊断...")
check_tun_device()
print("\n" + "="*50)
check_firewall()
print("\n" + "="*50)
check_network()
print("\n" + "="*50)
check_certificates()

print("\n✅ 诊断完成！如果发现问题，请参考以下解决方案:")
print("\n🔧 常见问题解决方案:")
print("1. TUN设备问题: 重新创建TUN设备")
print("2. 防火墙问题: 重新配置iptables规则")
print("3. 证书问题: 重新生成证书")
print("4. 网络问题: 检查IP转发设置")

## 🧹 清理环境（可选）

In [ ]:
# 清理函数
def cleanup():
    """清理OpenVPN环境"""
    print("🧹 清理OpenVPN环境...")
    
    # 停止服务器
    !pkill openvpn 2>/dev/null || true
    
    # 清理iptables规则
    !iptables -t nat -D POSTROUTING -s 10.8.0.0/24 -o eth0 -j MASQUERADE 2>/dev/null || true
    !iptables -D INPUT -p udp --dport 1194 -j ACCEPT 2>/dev/null || true
    !iptables -D FORWARD -i tun0 -j ACCEPT 2>/dev/null || true
    !iptables -D FORWARD -o tun0 -j ACCEPT 2>/dev/null || true
    
    # 删除TUN接口
    !ip link delete tun0 2>/dev/null || true
    
    # 清理文件
    !rm -rf /etc/openvpn/server/logs/* 2>/dev/null || true
    !rm -rf /tmp/openvpn-client 2>/dev/null || true
    
    print("✅ 环境清理完成")

# 注意：只有在需要时才运行清理
print("⚠️  警告：这将停止OpenVPN服务器并清理所有配置")
print("如果您想继续使用VPN，请不要运行此清理函数")
print("\n要清理环境，请取消注释下面的代码行:")
# cleanup()